# 20 - Integrate the REAL forecast wind with the graph model

Notebook 19 showed a real day-old wind forecast keeps almost all of the oracle gain for a graph-free model (58.53). This notebook asks the remaining question: once that forecast wind is in, does the **graph** add anything?

- Input window: observed wind + the observed-wind graph (as before).
- Next 24 hours: the real forecast wind as decoder features, **and** a graph built from that forecast wind (v3 used the actual future wind for this graph - an oracle). The recipe is verified against `dynamic_adj_om.npy` first.
- Models: static-GCN and the dynamic wind-aware graph, 5 seeds each; compared with the graph-free result from notebook 19.

Run notebooks 18 and 19 first. Run time about 50-60 min on the GPU.

In [1]:
# CELL 1 - imports + paths + constants
# HOW TO CHECK: prints the device (cuda) and that the two input files exist.
import os, sys, json, time, copy, pickle, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

BASE = r'D:\FYP\Eco-Twin'
V2 = os.path.join(BASE, 'v2_clean')
OUT = os.path.join(V2, 'ecotwin_outputs')
SCALER_DIR = os.path.join(OUT, 'scalers')
MODEL_DIR = os.path.join(OUT, 'models')
RESULTS_DIR = os.path.join(OUT, 'results')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

INPUT_WINDOW, HORIZON, BATCH_SIZE, LOSS_VOL_ALPHA = 48, 24, 16, 1.0
HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, DROPOUT, GCN_HOPS = 32, 1, 4, 0.2, 2
ALL_SEEDS = [42, 7, 123, 2024, 999, 2025, 77]
BASE_FEATS = ['pm2.5', 'pm10', 'no2', 'so2', 'co', 'o3', 'rh', 'at']
REST = ['pm_roll6_mean', 'pm_roll6_std', 'pm_roll24_mean', 'pm_roll24_std', 'pm_delta3',
        'city_mean_pm', 'city_max_pm', 'pm_minus_city',
        'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'mon_sin', 'mon_cos', 'is_weekend']
TIME_COLS = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'mon_sin', 'mon_cos', 'is_weekend']
PREVDAY_NPY = os.path.join(V2, 'openmeteo_prevday1_uv.npy')
ADJ_FC_NPY  = os.path.join(V2, 'dynamic_adj_prevday1.npy')
print('device:', DEVICE, '| prevday wind file:', os.path.exists(PREVDAY_NPY), '| ERA5 adjacency:', os.path.exists(os.path.join(V2, 'dynamic_adj_om.npy')))


device: cuda | prevday wind file: True | ERA5 adjacency: True


In [2]:
# CELL 2 - model (same architecture as nb14/nb15)
# HOW TO CHECK: runs silently.
class Net(nn.Module):
    def __init__(self, in_features, hidden_dim, gru_layers, attn_heads, forecast_horizon,
                 n_stations, dropout=0.2, n_fut=10, gcn_hops=2, graph_mode='dynamic', static_adj=None):
        super().__init__()
        H = hidden_dim
        self.input_proj = nn.Linear(in_features, H)
        self.gcn_lin = nn.ModuleList(nn.Linear(H, H) for _ in range(gcn_hops))
        self.gcn_norm = nn.ModuleList(nn.LayerNorm(H) for _ in range(gcn_hops))
        self.act = nn.ReLU()
        self.gru = nn.GRU(H, H, gru_layers, batch_first=True, dropout=dropout if gru_layers > 1 else 0.0)
        self.attn = nn.MultiheadAttention(H, attn_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(H)
        self.dropout = nn.Dropout(dropout)
        self.fut_enc = nn.Sequential(nn.Linear(n_fut, H), nn.ReLU(), nn.Linear(H, H))
        self.decoder = nn.Sequential(nn.Linear(H * 3, H), nn.ReLU(), nn.Dropout(dropout), nn.Linear(H, 1))
        self.anchor_gate = nn.Parameter(torch.linspace(1.0, 0.3, forecast_horizon))
        self.H, self.FH, self.graph_mode = H, forecast_horizon, graph_mode
        sa = torch.eye(n_stations) if static_adj is None else torch.from_numpy(static_adj.astype('float32'))
        self.register_buffer('static_adj', sa, persistent=False)

    @staticmethod
    def _n(A):
        return A / A.sum(-1, keepdim=True).clamp(min=1e-6)

    def forward(self, X, At, Afc, Fut, Anchor):
        B, Tin, n, _ = X.shape
        dev = X.device
        I = torch.eye(n, device=dev)
        h = self.act(self.input_proj(X))
        if self.graph_mode == 'none':
            for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
                h = self.act(nrm(lin(h))) + h
        else:
            if self.graph_mode == 'dynamic':
                A = self._n(At + I.view(1, 1, n, n))
            else:
                A = self._n((self.static_adj + I).view(1, 1, n, n)).expand(B, Tin, n, n)
            for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
                h = self.act(nrm(lin(torch.matmul(A, h)))) + h
        hf = h.permute(0, 2, 1, 3).reshape(B * n, Tin, self.H)
        gg, _ = self.gru(hf)
        gg = self.dropout(gg)
        aa, _ = self.attn(gg, gg, gg)
        ctx = self.attn_norm(aa + gg)[:, -1, :].reshape(B, n, self.H)
        ctx_rep = ctx.unsqueeze(1).expand(B, self.FH, n, self.H)
        if self.graph_mode == 'none':
            ctx_fc = ctx_rep
        elif self.graph_mode == 'dynamic':
            ctx_fc = torch.matmul(self._n(Afc + I.view(1, 1, n, n)), ctx_rep)
        else:
            ctx_fc = torch.matmul(self._n((self.static_adj + I).view(1, 1, n, n)), ctx_rep)
        fut_emb = self.fut_enc(Fut)
        resid = self.decoder(torch.cat([ctx_rep, ctx_fc, fut_emb], dim=-1)).squeeze(-1)
        return resid + Anchor.unsqueeze(1) * self.anchor_gate.view(1, self.FH, 1)


In [3]:
# CELL 3 - build the FUTURE-hour graph from the REAL forecast wind (no oracle)
# In v3 the graph used for the 24 forecast hours was built from the actual future wind. Here it is built from the
# day-old forecast instead. First we prove this recipe is identical to the one that made dynamic_adj_om.npy.
# HOW TO CHECK: "max abs diff" vs the saved ERA5 adjacency should be ~0 (< 1e-4) and edge counts equal.
# If it is not, STOP and tell me - the recipe would be different from the one the model was trained with.
df_st = pd.read_excel(os.path.join(V2, 'Delhi_list_v2.xlsx'))
df_st.columns = df_st.columns.str.strip().str.lower()
lats = df_st['latitude'].values.astype(float)
lons = df_st['longitude'].values.astype(float)
N = len(lats)

def haversine_matrix(la, lo):
    R = 6371.0
    p1, p2 = np.radians(la)[:, None], np.radians(la)[None, :]
    dphi, dl = p2 - p1, (np.radians(lo)[None, :] - np.radians(lo)[:, None])
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

dist = haversine_matrix(lats, lons); np.fill_diagonal(dist, 0.0)
within = (dist > 0) & (dist <= 12.0)
d_east = (lons[None, :] - lons[:, None]) * np.cos(np.radians(lats))[:, None]
d_north = lats[None, :] - lats[:, None]
nrm = np.hypot(d_east, d_north)
dir_u = np.where(nrm > 0, d_east / np.where(nrm > 0, nrm, 1), 0.0)
dir_v = np.where(nrm > 0, d_north / np.where(nrm > 0, nrm, 1), 0.0)

def adj_from_wind(uv, block=1500):
    T = uv.shape[0]; out = np.zeros((T, N, N), dtype=np.float32)
    for s in range(0, T, block):
        u, v = uv[s:s + block, :, 0].astype(np.float64), uv[s:s + block, :, 1].astype(np.float64)
        align = u[:, :, None] * dir_u[None] + v[:, :, None] * dir_v[None]
        A = np.where(within[None] & (align > 0), align / (dist[None] + 1e-6), 0.0).astype(np.float32)
        rs = A.sum(2, keepdims=True); rs[rs == 0] = 1
        out[s:s + block] = A / rs
    return out

era5_uv = np.load(os.path.join(V2, 'openmeteo_reanalysis_uv.npy')).astype(np.float32)
ref = np.load(os.path.join(V2, 'dynamic_adj_om.npy'), mmap_mode='r')
mine = adj_from_wind(era5_uv)
print('recipe check vs dynamic_adj_om.npy:  max abs diff', float(np.abs(mine - ref).max()),
      '| edges', int((mine > 0).sum()), 'vs', int((np.asarray(ref) > 0).sum()))
del mine

prev_uv = np.load(PREVDAY_NPY).astype(np.float32)
adj_fc = adj_from_wind(prev_uv)
np.save(ADJ_FC_NPY, adj_fc)
print('saved', ADJ_FC_NPY, adj_fc.shape, '| mean edges/hour (forecast)', round(float((adj_fc > 0).sum((1, 2)).mean()), 1),
      '| (ERA5)', round(float((np.asarray(ref) > 0).sum((1, 2)).mean()), 1))
del adj_fc, era5_uv, prev_uv


recipe check vs dynamic_adj_om.npy:  max abs diff 1.4901161193847656e-08 | edges 5116599 vs 5116599
saved D:\FYP\Eco-Twin\v2_clean\dynamic_adj_prevday1.npy (19921, 37, 37) | mean edges/hour (forecast) 258.1 | (ERA5) 256.8


In [4]:
# CELL 4 - dataset + data builder
# Inputs for ALL variants: 23 CPCB features + OBSERVED wind (ERA5) over the 48 past hours = 26 features.
# What differs between variants is only the wind the decoder is shown for the 24 FUTURE hours:
#   observed_only : no future wind (calendar only)
#   real_forecast : the day-old forecast (previous_day1) - what a real system could have
#   oracle        : the actual future wind (ERA5) - impossible in practice, upper bound
# Input window: observed wind + observed-wind graph. Future 24 h: REAL forecast wind features + forecast-wind graph.
# HOW TO CHECK: runs silently (the build itself happens in the next cell).
class DS(Dataset):
    def __init__(self, ctx, starts, weights=None):
        self.c = ctx
        self.s = np.asarray(starts, np.int64)
        self.w = (weights if weights is not None else np.ones(len(self.s), np.float32)).astype(np.float32)

    def __len__(self):
        return len(self.s)

    def __getitem__(self, k):
        c = self.c
        g = int(self.s[k]); i1 = g + INPUT_WINDOW; i2 = i1 + HORIZON
        return (torch.from_numpy(c['Xs'][g:i1].copy()),
                torch.from_numpy(c['adj'][g:i1].copy()),
                torch.from_numpy(c['adj_fc'][i1:i2].copy()),
                torch.from_numpy(np.ascontiguousarray(c['Xs_fut'][i1:i2][:, :, c['FUT_IDX']])),
                torch.from_numpy(c['Yl'][i1 - 1].copy()),
                torch.from_numpy(c['Yl'][i1:i2].copy()),
                torch.tensor(self.w[k], dtype=torch.float32))


def build_base():
    df = pd.read_csv(os.path.join(V2, 'EcoTwin_Final_Dataset.csv'), parse_dates=['Timestamp'])
    df['Station'] = df['Station'].str.strip()
    df = df.sort_values(['Station', 'Timestamp']).reset_index(drop=True)
    df_st = pd.read_excel(os.path.join(V2, 'Delhi_list_v2.xlsx'))
    df_st.columns = df_st.columns.str.strip().str.lower()
    df_st['station'] = df_st['station'].str.strip()
    names = df_st['station'].tolist()
    N = len(names)
    T = len(np.array(sorted(df['Timestamp'].unique())))
    era5 = np.load(os.path.join(V2, 'openmeteo_reanalysis_uv.npy')).astype(np.float32)
    era5_spd = np.hypot(era5[..., 0], era5[..., 1])
    prevday = np.load(PREVDAY_NPY).astype(np.float32)
    assert prevday.shape == era5.shape, (prevday.shape, era5.shape)
    prev_spd = np.hypot(prevday[..., 0], prevday[..., 1])
    adj = np.load(os.path.join(V2, 'dynamic_adj_om.npy')).astype(np.float32)        # observed-wind graph, input window
    adj_fc = np.load(ADJ_FC_NPY).astype(np.float32)                                  # forecast-wind graph, 24 future hours
    static_adj = np.load(os.path.join(V2, 'static_adj_matrix.npy')).astype(np.float32)
    np.fill_diagonal(static_adj, 0.0)
    assert adj.shape == (T, N, N) and adj_fc.shape == (T, N, N), (adj.shape, adj_fc.shape)

    g = df.groupby('Station', sort=False)
    df['pm_roll6_mean'] = g['pm2.5'].transform(lambda s: s.rolling(6, min_periods=1).mean())
    df['pm_roll6_std'] = g['pm2.5'].transform(lambda s: s.rolling(6, min_periods=1).std()).fillna(0.0)
    df['pm_roll24_mean'] = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).mean())
    df['pm_roll24_std'] = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).std()).fillna(0.0)
    df['pm_delta3'] = g['pm2.5'].transform(lambda s: s - s.shift(3)).fillna(0.0)
    tsg = df.groupby('Timestamp', sort=False)['pm2.5']
    df['city_mean_pm'] = tsg.transform('mean')
    df['city_max_pm'] = tsg.transform('max')
    df['pm_minus_city'] = df['pm2.5'] - df['city_mean_pm']
    hh = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month
    df['hour_sin'] = np.sin(2 * np.pi * hh / 24); df['hour_cos'] = np.cos(2 * np.pi * hh / 24)
    df['dow_sin'] = np.sin(2 * np.pi * dow / 7); df['dow_cos'] = np.cos(2 * np.pi * dow / 7)
    df['mon_sin'] = np.sin(2 * np.pi * (mon - 1) / 12); df['mon_cos'] = np.cos(2 * np.pi * (mon - 1) / 12)
    df['is_weekend'] = (dow >= 5).astype(float)

    Xr = np.zeros((T, N, len(BASE_FEATS + REST)), dtype=np.float32)
    for fi, col in enumerate(BASE_FEATS + REST):
        wide = df.pivot_table(index='Timestamp', columns='Station', values=col).reindex(columns=names)
        Xr[:, :, fi] = np.nan_to_num(wide.values.astype(np.float32))

    FEATURE_COLS = BASE_FEATS + ['u_wind', 'v_wind', 'wspd'] + REST
    X_raw = np.concatenate([Xr[:, :, :8], era5[:, :, 0:1], era5[:, :, 1:2], era5_spd[:, :, None], Xr[:, :, 8:]],
                           axis=2).astype(np.float32)
    F = len(FEATURE_COLS)
    Y_raw = X_raw[:, :, 0].copy()
    TIME_IDX = [FEATURE_COLS.index(c) for c in TIME_COLS]
    U_IDX, V_IDX, W_IDX = (FEATURE_COLS.index(c) for c in ['u_wind', 'v_wind', 'wspd'])

    train_end, val_end = int(T * 0.70), int(T * 0.85)

    def starts(lo, hi):
        return np.arange(lo, lo + (hi - lo) - (INPUT_WINDOW + HORIZON) + 1)
    tr_s, va_s, te_s = starts(0, train_end), starts(train_end, val_end), starts(val_end, T)

    tr_mask = np.zeros(T, bool); tr_mask[:train_end] = True
    feat_scaler = StandardScaler().fit(X_raw[tr_mask].reshape(-1, F))
    Xs = feat_scaler.transform(X_raw.reshape(-1, F)).reshape(T, N, F).astype(np.float32)

    # forecast wind, standardised with the SAME scaler statistics as the observed wind columns
    Xs_fc = Xs.copy()
    for idx, raw in [(U_IDX, prevday[..., 0]), (V_IDX, prevday[..., 1]), (W_IDX, prev_spd)]:
        Xs_fc[:, :, idx] = (raw - feat_scaler.mean_[idx]) / feat_scaler.scale_[idx]

    log_scaler = StandardScaler().fit(np.log1p(np.clip(Y_raw[tr_mask].reshape(-1, 1), 0, None)))

    def to_log(a):
        return log_scaler.transform(np.log1p(np.clip(a, 0, None)).reshape(-1, 1)).reshape(a.shape).astype(np.float32)

    def from_log(a):
        return np.expm1(log_scaler.inverse_transform(a.reshape(-1, 1)).reshape(a.shape))

    Yl = to_log(Y_raw)
    swing = np.array([(Y_raw[s + INPUT_WINDOW:s + INPUT_WINDOW + HORIZON].max(0) -
                       Y_raw[s + INPUT_WINDOW:s + INPUT_WINDOW + HORIZON].min(0)).mean() for s in tr_s], np.float32)
    med = float(np.median(swing))
    tr_w = np.clip(1.0 + LOSS_VOL_ALPHA * (swing / (med + 1e-6)), 0.5, 4.0).astype(np.float32)
    tr_w *= len(tr_w) / tr_w.sum()

    ctx = dict(Xs=Xs, Xs_fc=Xs_fc, Xs_fut=Xs, Yl=Yl, adj=adj, adj_fc=adj_fc, static_adj=static_adj, F=F, N=N, T=T,
               TIME_IDX=TIME_IDX, U_IDX=U_IDX, V_IDX=V_IDX, W_IDX=W_IDX, FUT_IDX=TIME_IDX, n_fut=len(TIME_IDX),
               FEATURE_COLS=FEATURE_COLS, from_log=from_log, era5_spd=era5_spd, te_starts=te_s,
               tr_s=tr_s, va_s=va_s, te_s=te_s, tr_w=tr_w, era5=era5, prevday=prevday)
    print(f'[build] F={F} windows train/val/test {len(tr_s)}/{len(va_s)}/{len(te_s)}', flush=True)
    return ctx


def set_variant(ctx, name):
    if name == 'observed_only':
        ctx['FUT_IDX'], ctx['Xs_fut'] = ctx['TIME_IDX'], ctx['Xs']
    elif name == 'real_forecast':
        ctx['FUT_IDX'], ctx['Xs_fut'] = ctx['TIME_IDX'] + [ctx['U_IDX'], ctx['V_IDX'], ctx['W_IDX']], ctx['Xs_fc']
    elif name == 'oracle':
        ctx['FUT_IDX'], ctx['Xs_fut'] = ctx['TIME_IDX'] + [ctx['U_IDX'], ctx['V_IDX'], ctx['W_IDX']], ctx['Xs']
    else:
        raise ValueError(name)
    ctx['n_fut'] = len(ctx['FUT_IDX'])
    ctx['tr'] = DataLoader(DS(ctx, ctx['tr_s'], ctx['tr_w']), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    ctx['va'] = DataLoader(DS(ctx, ctx['va_s']), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    ctx['te'] = DataLoader(DS(ctx, ctx['te_s']), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return ctx


In [5]:
# CELL 5 - train / predict / metric helpers
# HOW TO CHECK: runs silently.
def make_model(ctx, mode):
    return Net(ctx['F'], HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, HORIZON, ctx['N'], DROPOUT,
               ctx['n_fut'], GCN_HOPS, graph_mode=mode, static_adj=ctx['static_adj']).to(DEVICE)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    P, A = [], []
    for X, At, Afc, Fut, anc, y, w in loader:
        P.append(model(X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE), Fut.to(DEVICE), anc.to(DEVICE)).cpu().numpy())
        A.append(y.numpy())
    return np.concatenate(P), np.concatenate(A)


def metrics(ctx, pl, al):
    pr, ar = ctx['from_log'](pl), ctx['from_log'](al)
    return dict(rmse=float(np.sqrt(mean_squared_error(ar.ravel(), pr.ravel()))),
                mae=float(mean_absolute_error(ar.ravel(), pr.ravel())),
                r2=float(r2_score(ar.ravel(), pr.ravel())))


def weighted_huber(pred, target, w, delta=1.0):
    absr = (pred - target).abs()
    quad = torch.clamp(absr, max=delta)
    per = (0.5 * quad ** 2 + delta * (absr - quad)).mean(dim=(1, 2))
    return (per * w).mean() / w.mean().clamp(min=1e-6)


def run_epoch(model, loader, opt=None, sched=None):
    tr = opt is not None
    model.train(tr)
    tot = n = 0
    for X, At, Afc, Fut, anc, y, w in loader:
        X, At, Afc = X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE)
        Fut, anc, y, w = Fut.to(DEVICE), anc.to(DEVICE), y.to(DEVICE), w.to(DEVICE)
        with torch.set_grad_enabled(tr):
            out = model(X, At, Afc, Fut, anc)
            loss = weighted_huber(out, y, w if tr else torch.ones_like(w))
        if tr:
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if sched:
                sched.step()
        tot += loss.item() * X.size(0); n += X.size(0)
    return tot / n


def train_one(ctx, mode, seed, max_epochs=60, patience=12, lr=1e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    model = make_model(ctx, mode)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps = len(ctx['tr']) * max_epochs
    warm = max(1, int(0.05 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s / warm if s < warm else 0.5 * (1 + np.cos(np.pi * (s - warm) / max(1, steps - warm))))
    best, best_state, ctr = 1e9, None, 0
    for ep in range(1, max_epochs + 1):
        t0 = time.time()
        trl = run_epoch(model, ctx['tr'], opt, sched)
        vl = run_epoch(model, ctx['va'])
        if vl < best - 1e-5:
            best, best_state, ctr = vl, copy.deepcopy(model.state_dict()), 0
        else:
            ctr += 1
        if ep % 4 == 0 or ctr >= patience:
            print(f'    ep {ep:>2} train {trl:.4f} val {vl:.4f} ({time.time() - t0:.0f}s) best {best:.4f}', flush=True)
        if ctr >= patience:
            break
    model.load_state_dict(best_state)
    return model


In [6]:
# CELL 6 - TRAIN: static-graph and dynamic wind-graph WA-TGCN, both with REAL forecast wind, 5 seeds each
# (~50-60 min; keep this tab open, no other GPU jobs). The graph-free result (58.53) comes from notebook 19.
# HOW TO CHECK: first line prints the build; each seed prints "test RMSE" (expect roughly 57-65 per seed).
# Progress is saved after every seed.
ctx = build_base()
set_variant(ctx, 'real_forecast')
print('n_fut =', ctx['n_fut'], '(10 = calendar + forecast wind)')
seeds5 = ALL_SEEDS[:5]
preds, actual, progress = {}, None, {}
for mode in ['static', 'dynamic']:
    ps = []
    for sd in seeds5:
        print(f'\n[{mode} + real forecast wind] seed {sd}', flush=True)
        m = train_one(ctx, mode, sd)
        p, a = predict(m, ctx['te']); ps.append(p); actual = a
        r = metrics(ctx, p, a)['rmse']
        print(f'  -> seed {sd} test RMSE {r:.2f}', flush=True)
        progress.setdefault(mode, []).append(r)
        json.dump(progress, open(os.path.join(RESULTS_DIR, 'v3_integration_progress.json'), 'w'), indent=1)
        del m; torch.cuda.empty_cache()
    preds[mode] = ps
print('\nTRAINING DONE - now run the next cell')


[build] F=26 windows train/val/test 13873/2917/2918
n_fut = 10 (10 = calendar + forecast wind)

[static + real forecast wind] seed 42
    ep  4 train 0.1247 val 0.1411 (19s) best 0.1411
    ep  8 train 0.1054 val 0.1478 (19s) best 0.1411
    ep 12 train 0.0953 val 0.1540 (20s) best 0.1411
    ep 16 train 0.0896 val 0.1625 (21s) best 0.1411
  -> seed 42 test RMSE 61.79

[static + real forecast wind] seed 7
    ep  4 train 0.1235 val 0.1362 (21s) best 0.1362
    ep  8 train 0.1020 val 0.1602 (20s) best 0.1362
    ep 12 train 0.0932 val 0.1527 (19s) best 0.1362
    ep 16 train 0.0881 val 0.1643 (20s) best 0.1362
  -> seed 7 test RMSE 58.96

[static + real forecast wind] seed 123
    ep  4 train 0.1224 val 0.1435 (21s) best 0.1394
    ep  8 train 0.1032 val 0.1746 (19s) best 0.1394
    ep 12 train 0.0938 val 0.1766 (21s) best 0.1394
    ep 14 train 0.0907 val 0.1801 (20s) best 0.1394
  -> seed 123 test RMSE 58.82

[static + real forecast wind] seed 2024
    ep  4 train 0.1234 val 0.1539 (1

In [7]:
# CELL 7 - results + save. Compares against the graph-free real-forecast result from notebook 19.
# HOW TO CHECK: prints the table. Differences under ~0.5 are inside the seed noise.
ar = ctx['from_log'](actual)
sw = ar.max(1) - ar.min(1)
q33, q66 = np.percentile(sw, [33, 66])
vreg = {'Stable': sw <= q33, 'Moderate': (sw > q33) & (sw <= q66), 'Volatile': sw > q66}

res = {}
for mode, ps in preds.items():
    per_seed = [metrics(ctx, p, actual)['rmse'] for p in ps]
    ens = metrics(ctx, np.mean(ps, axis=0), actual)
    pr = ctx['from_log'](np.mean(ps, axis=0))
    res[mode] = dict(ens=ens, per_seed_rmse=per_seed, mean_seed_rmse=float(np.mean(per_seed)),
                     std_seed_rmse=float(np.std(per_seed)),
                     by_horizon={f'+{h+1}h': float(np.sqrt(((ar[:, h, :] - pr[:, h, :]) ** 2).mean())) for h in [0, 5, 11, 17, 23]},
                     by_volatility={k: float(np.sqrt(((ar.transpose(0, 2, 1)[m] - pr.transpose(0, 2, 1)[m]) ** 2).mean()))
                                    for k, m in vreg.items()})

ref = json.load(open(os.path.join(RESULTS_DIR, 'v3_realfc_ablation.json')))['results']['real_forecast']
res['none'] = dict(ens=ref['ens'], per_seed_rmse=ref['per_seed_rmse'], mean_seed_rmse=ref['mean_seed_rmse'],
                   std_seed_rmse=ref['std_seed_rmse'], by_horizon=ref['by_horizon'], by_volatility=ref['by_volatility'])
json.dump({'note': 'WA-TGCN with REAL day-old forecast wind features; dynamic graph uses observed-wind graph for the input '
                   'window and a forecast-wind graph for the 24 future hours. none = notebook 19 real_forecast.',
           'results': res}, open(os.path.join(RESULTS_DIR, 'v3_integration.json'), 'w'), indent=2)

names = {'none': 'graph-free + real forecast wind', 'static': 'static-GCN + real forecast wind',
         'dynamic': 'WA-TGCN (wind graph) + real forecast wind'}
print('=== does the graph add anything once REAL forecast wind is in? (test, 5-seed ensembles) ===')
print(f"{'model':44}{'ens RMSE':>10}{'MAE':>8}{'R2':>8}{'seed mean':>11}{'seed std':>10}")
for k in ['none', 'static', 'dynamic']:
    r = res[k]
    print(f"{names[k]:44}{r['ens']['rmse']:>10.2f}{r['ens']['mae']:>8.2f}{r['ens']['r2']:>8.3f}"
          f"{r['mean_seed_rmse']:>11.2f}{r['std_seed_rmse']:>10.2f}")
print(f"\nwind graph vs static graph : {res['static']['ens']['rmse'] - res['dynamic']['ens']['rmse']:+.2f}")
print(f"wind graph vs graph-free   : {res['none']['ens']['rmse'] - res['dynamic']['ens']['rmse']:+.2f}")
print('\nby horizon (+1h / +6h / +12h / +18h / +24h):')
for k in ['none', 'static', 'dynamic']:
    print(f"  {names[k]:44}" + ''.join(f'{x:>8.1f}' for x in res[k]['by_horizon'].values()))
print('by volatility (Stable / Moderate / Volatile):')
for k in ['none', 'static', 'dynamic']:
    print(f"  {names[k]:44}" + ''.join(f'{x:>8.1f}' for x in res[k]['by_volatility'].values()))
print('\nper-seed test RMSE:')
for k in ['none', 'static', 'dynamic']:
    print(f"  {names[k]:44}", [round(x, 2) for x in res[k]['per_seed_rmse']])
print('\nsaved v3_integration.json')


=== does the graph add anything once REAL forecast wind is in? (test, 5-seed ensembles) ===
model                                         ens RMSE     MAE      R2  seed mean  seed std
graph-free + real forecast wind                  58.53   38.17   0.662      60.35      2.11
static-GCN + real forecast wind                  58.62   38.27   0.661      60.76      1.59
WA-TGCN (wind graph) + real forecast wind        59.51   38.52   0.651      61.14      1.28

wind graph vs static graph : -0.89
wind graph vs graph-free   : -0.98

by horizon (+1h / +6h / +12h / +18h / +24h):
  graph-free + real forecast wind                 41.2    55.6    60.6    62.6    62.5
  static-GCN + real forecast wind                 41.8    55.4    60.5    62.9    63.0
  WA-TGCN (wind graph) + real forecast wind       40.4    56.3    61.5    64.1    64.2
by volatility (Stable / Moderate / Volatile):
  graph-free + real forecast wind                 27.9    41.4    87.5
  static-GCN + real forecast wind            